In [1]:
import httpx
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
import time
import json
import psycopg
import os
import sys
import numpy as np
from dotenv import load_dotenv

sys.path.append(os.path.abspath('./src'))
import db_functions as dbf
load_dotenv()
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST", "localhost")
port = os.getenv("DB_PORT", "5432")
dbname = os.getenv("DB_NAME")
conn_str = f"postgresql://{user}:{password}@{host}:{port}/{dbname}"
api_key = os.getenv("API_KEY")
api_url = 'https://api.stratz.com/graphql'
headers = {
    'User-Agent': 'STRATZ_API',
    "Authorization": f"Bearer {api_key}"
}

In [ ]:
#TODO before use: integrate the steamAccount gathering logic from later code block
query = """
    query($id: Long!) {
      match(id: $id) {
        id
        tournamentId
        tournamentRound
        leagueId
        radiantTeamId
        direTeamId
        seriesId
        gameVersionId
        regionId
        clusterId
        didRadiantWin
        startDateTime
        endDateTime
        durationSeconds
        firstBloodTime
        towerStatusRadiant
        towerStatusDire
        barracksStatusRadiant
        barracksStatusDire
        rank
        actualRank
        averageRank
        averageImp
        bracket
        analysisOutcome
        topLaneOutcome
        midLaneOutcome
        bottomLaneOutcome
        predictedOutcomeWeight
        pickBans {
          isPick
          heroId
          order
          isRadiant
        }
        chatEvents {
          time
          type
          fromHeroId
          toHeroId
          value
          pausedTick
          isRadiant
        }
        predictedWinRates
        winRates
        radiantNetworthLeads
        radiantExperienceLeads
        radiantKills
        direKills
        towerDeaths {
          time
          npcId
          isRadiant
          attacker
        }
        towerStatus {
          towers {
            npcId
            hp
          }
          outposts {
            npcId
            isControlledByRadiant
            isRadiantSide
          }
        }
        players {
          heroId
          steamAccountId
          isRadiant
          isVictory
          variant
          imp
          lane
          position
          networth
          goldPerMinute
          goldSpent
          towerDamage
          heroDamage
          intentionalFeeding
          stats {
            impPerMinute
            goldPerMinute
            networthPerMinute
            experiencePerMinute
            towerDamagePerMinute
            campStack
            deathEvents {
              time
              attacker
              isDieBack
            }
            farmDistributionReport {
              creepLocation {
                id
                gold
              }
              neutralLocation {
                id
                gold
              }
              ancientLocation {
                id
                gold
              }
              buildings {
                id
                gold
              }
              bountyGold {
                id
                gold
              }
              other {
                id
                gold
              }
              buyBackGold
            }
            matchPlayerBuffEvent {
              time
              abilityId
              itemId
              stackCount
            }
            inventoryReport {
              item0 {
                itemId
              }
              item1 {
                itemId
              }
              item2 {
                itemId
              }
              item3 {
                itemId
              }
              item4 {
                itemId
              }
              item5 {
                itemId
              }
              neutral0 {
                itemId
              }
            }
            itemPurchases {
              time
              itemId
            }
            courierKills {
              time
            }
            runes {
              time
              rune
              action
              positionX
              positionY
            }
            wards {
              time
              type
              positionX
              positionY
            }
            wardDestruction {
              time
              gold
              isWard
            }
          }
        }
      }
    }
    """

In [ ]:
## Preparing column lists for parsing matches to sql
cols_to_include = [
    'id',
    'tournamentId',
    'tournamentRound',
    'leagueId',
    'radiantTeamId',
    'direTeamId',
    'seriesId',
    'gameVersionId',
    'regionId',
    'clusterId',
    'didRadiantWin',
    'startDateTime',
    'endDateTime',
    'durationSeconds',
    'firstBloodTime',
    'towerStatusRadiant',
    'towerStatusDire',
    'barracksStatusRadiant',
    'barracksStatusDire',
    'rank',
    'actualRank',
    'averageRank',
    'averageImp',
    'bracket',
    'analysisOutcome',
    'topLaneOutcome',
    'midLaneOutcome',
    'bottomLaneOutcome',
    'predictedOutcomeWeight'
]
cols_to_keep = [
    'heroId',
    'steamAccountId',
    'partId',
    'isRadiant',
    'isVictory',
    'variant',
    'imp',
    'lane',
    'position',
    'networth',
    'goldPerMinute',
    'goldSpent',
    'towerDamage',
    'heroDamage',
    'intentionalFeeding'
]
performance_metrics_columns = [
    'match_id',
    'hero_id',
    'minute',
    'gold_per_minute',
    'networth_per_minute',
    'experience_per_minute',
    'tower_damage_per_minute',
    'camp_stack'
]

In [ ]:
# with psycopg.connect(conn_str) as conn:
#     with conn.cursor() as cur:
#         result = cur.execute("SELECT table_name FROM information_schema.tables").fetchall()
#         for table_name in result:
#             if table_name[0].startswith('match_'):
#                 cur.execute(f'DROP TABLE {table_name[0]}')

In [15]:
## Parsing matches to dataframes and to sql tables
with psycopg.connect(conn_str) as conn:
    with conn.cursor() as cur:
        match_id_tups = cur.execute("SELECT match_id FROM matches WHERE start_date >= '2025-02-21 00:00:00' ORDER BY start_date ASC;").fetchall()
        match_ids = [tup[0] for tup in match_id_tups]
        parsed_id_tups = cur.execute("SELECT id FROM match_details").fetchall()
        parsed_ids = [tup[0] for tup in parsed_id_tups]
        unfinished_match_ids = []
        for match_id in match_ids:
            if match_id not in parsed_ids:
                unfinished_match_ids.append(match_id)


In [16]:
len(unfinished_match_ids)

380

In [ ]:
#TODO before use: integrate the steamAccount gathering logic from later code block
counter = 0
for match_id in unfinished_match_ids:
    start_time = time.time()
    variables = {'id': match_id}
    result = dbf.query_stratz(query, headers=headers, api_url=api_url, variables=variables)
    result_json = result['data']['match']
    #TODO: find first blood team by looking for chat event type 5
    #TODO: both hero IDs present: tip
    #TODO: find out what is roshan kills etc. from an actual replay

    #TODO: add buybackgold to players_df / match_players SQL table
    filtered_match_details = {k: result_json[k] for k in cols_to_include}
    df_match_details = pd.DataFrame([filtered_match_details])
    df_pickbans = pd.DataFrame(result_json['pickBans'])
    df_pickbans.insert(0, 'match_id', result_json['id'])
    try:
        df_chatevents = pd.json_normalize(result_json['chatEvents'])
    except:
        result = dbf.query_stratz(query, headers=headers, api_url=api_url, variables=variables)
        result_json = result['data']['match']
        try:
            df_chatevents = pd.json_normalize(result_json['chatEvents'])
        except:
            continue
    df_chatevents.insert(0, 'match_id', result_json['id'])
    try:
        df_predicted_win_rates = pd.DataFrame({
            'match_id': result_json['id'],
            'predicted_win_rate': result_json['predictedWinRates']
        })
    except:
        result = dbf.query_stratz(query, headers=headers, api_url=api_url, variables=variables)
        result_json = result['data']['match']
        try:
            df_predicted_win_rates = pd.DataFrame({
            'match_id': result_json['id'],
            'predicted_win_rate': result_json['predictedWinRates']
            })
        except:
            continue
    df_win_rates = pd.DataFrame({
        'match_id': result_json['id'],
        'win_rates': result_json['winRates'], 
    })
    df_kills = pd.DataFrame({
        'match_id': result_json['id'],
        'radiant_kills': result_json['radiantKills'],
        'dire_kills': result_json['direKills']
    })
    df_leads = pd.DataFrame({
        'match_id': result_json['id'],
        'radiant_networth_leads': result_json['radiantNetworthLeads'],
        'radiant_experience_leads': result_json['radiantExperienceLeads']
    })
    df_tower_deaths = pd.json_normalize(result_json['towerDeaths'])
    df_tower_deaths.insert(0, 'match_id', result_json['id'])
    snapshots = []
    tower_updates = []
    outpost_updates = []
    for i, snapshot in enumerate(result_json['towerStatus']):
        snapshot_id = f"{result_json['id']}_{i}"
        snapshots.append({
            'snapshot_id': snapshot_id,
            'match_id': result_json['id'],
            'order_index': i 
        })
        for t in snapshot['towers']:
            tower_updates.append({
                'snapshot_id': snapshot_id,
                'npc_id': t['npcId'],
                'hp': t['hp']
            })
            
        for o in snapshot['outposts']:
            outpost_updates.append({
                'snapshot_id': snapshot_id,
                'npc_id': o['npcId'],
                'is_radiant_controlled': o['isControlledByRadiant'],
                'is_radiant_side': o['isRadiantSide']
            })
    df_snapshots = pd.DataFrame(snapshots)
    df_tower_updates = pd.DataFrame(tower_updates)
    df_outpost_updates = pd.DataFrame(outpost_updates)
    df_players = pd.json_normalize(result_json['players'])
    df_players = df_players[cols_to_keep].rename('steamAccountId': 'steam_account_id') ## I forgot to save these initially so I had to rename for later use
    df_players.insert(0, 'match_id', result_json['id'])
    df_performance_metrics = pd.DataFrame(columns=performance_metrics_columns)
    df_death_events = pd.DataFrame(columns=['match_id', 'hero_id', 'time', 'attacker', 'isDieBack'])
    df_farm = pd.DataFrame(columns=['match_id', 'hero_id', 'source_type', 'id',	'gold'])
    df_courier_kills = pd.DataFrame(columns=['match_id', 'hero_id', 'time'])
    df_runes = pd.DataFrame(columns=['match_id', 'hero_id', 'time', 'rune', 'action', 'positionX', 'positionY'])
    df_wards = pd.DataFrame(columns=[
        'match_id',
        'hero_id',
        'time',
        'type',
        'positionX',
        'positionY'
    ])
    df_ward_destructions = pd.DataFrame(columns=[
        'match_id',
        'hero_id',
        'time',
        'gold',
        'isWard'
    ])
    df_inventory_reports = pd.DataFrame(columns=[
        'match_id',
        'hero_id',
        'minute',
        'item0_id',
        'item1_id',
        'item2_id',
        'item3_id',
        'item4_id',
        'item5_id',
        'neutral0_id',
    ])
    df_purchases = pd.DataFrame(columns=['match_id', 'hero_id', 'time', 'itemId'])
    df_buffs = pd.DataFrame(columns=[
        'match_id',
        'hero_id',
        'time',
        'abilityId',
        'itemId',
        'stackCount'
    ])
    df_imp_per_minute = pd.DataFrame(columns=['match_id', 'hero_id', 'imp_per_minute'])
    for idx, player in enumerate(result_json['players']):
        stats = player['stats'] 

        df_ir = pd.json_normalize(stats['inventoryReport'])
        existing_item_cols = [c for c in df_ir.columns if '.itemId' in c]
        df_ir = df_ir[existing_item_cols].copy()
        df_ir = df_ir.rename(columns=lambda x: x.replace('.itemId', '_id'))
        df_ir.insert(0, 'hero_id', player['heroId'])
        df_ir.insert(0, 'match_id', result_json['id'])
        df_inventory_reports = pd.concat([df_inventory_reports, df_ir])
        df_ipm = pd.DataFrame({
            'match_id': result_json['id'],
            'hero_id': player['heroId'],
            'imp_per_minute': stats['impPerMinute']
        }) 
        df_imp_per_minute = pd.concat([df_imp_per_minute, df_ipm])
        df_pm = pd.DataFrame({
            'match_id': result_json['id'],
            'hero_id': player['heroId'],
            'minute': range(len(stats['networthPerMinute'])),
            'gold_per_minute': pd.Series(stats['goldPerMinute']),
            'networth_per_minute': pd.Series(stats['networthPerMinute']),
            'experience_per_minute': pd.Series(stats['experiencePerMinute']),
            'tower_damage_per_minute': pd.Series(stats['towerDamagePerMinute']),
            'camp_stack': pd.Series(stats['campStack'])
        })
        df_performance_metrics = pd.concat([df_performance_metrics, df_pm])
        df_matchid_heroid = pd.DataFrame({
            'match_id': result_json['id'],
            'hero_id': player['heroId']
        }, index=range(len(stats['deathEvents'])))
        df_de = pd.DataFrame(stats['deathEvents'])
        df_death_events = pd.concat([df_death_events, pd.concat([df_matchid_heroid, df_de], axis=1)])
        rows = []
        for category, content in stats['farmDistributionReport'].items():
            if isinstance(content, list):
                for entry in content:
                    # Copy entry so we don't modify the original JSON
                    row = entry.copy()
                    row['source_type'] = category
                    rows.append(row)
            elif isinstance(content, dict):
                row = content.copy()
                row['source_type'] = category
                rows.append(row)
        df_f = pd.DataFrame(rows)
        df_f.insert(0, 'hero_id', player['heroId'])
        df_f.insert(0, 'match_id', result_json['id'])
        df_farm = pd.concat([df_farm, df_f])
        buff_rows = []
        for buff in stats['matchPlayerBuffEvent']:
            buff_rows.append({
                'time': buff['time'],
                'item_id': buff['itemId'],
                'ability_id': buff['abilityId']
            })
        df_b = pd.DataFrame(buff_rows)
        df_b.insert(0, 'hero_id', player['heroId'])
        df_b.insert(0, 'match_id', result_json['id'])
        df_buffs = pd.concat([df_buffs, df_b])

        df_p = pd.DataFrame(stats['itemPurchases'])
        df_p.insert(0, 'hero_id', player['heroId'])
        df_p.insert(0, 'match_id', result_json['id'])
        df_purchases = pd.concat([df_purchases, df_p])
        df_c = pd.DataFrame(stats['courierKills'])
        df_c.insert(0, 'hero_id', player['heroId'])
        df_c.insert(0, 'match_id', result_json['id'])
        df_courier_kills = pd.concat([df_courier_kills, df_c])
        df_r = pd.DataFrame(stats['runes'])
        df_r.insert(0, 'hero_id', player['heroId'])
        df_r.insert(0, 'match_id', result_json['id'])
        df_runes = pd.concat([df_runes, df_r])
        df_w = pd.DataFrame(stats['wards'])
        df_w.insert(0, 'hero_id', player['heroId'])
        df_w.insert(0, 'match_id', result_json['id'])
        df_wards = pd.concat([df_wards, df_w])
        df_wd = pd.DataFrame(stats['wardDestruction'])
        df_wd.insert(0, 'hero_id', player['heroId'])
        df_wd.insert(0, 'match_id', result_json['id'])
        df_ward_destructions = pd.concat([df_ward_destructions, df_wd])
    df_death_events = df_death_events.reset_index().drop(['index'], axis=1)
    first_layer_dfs = [
        df_match_details,
        df_pickbans,
        df_chatevents,
        df_predicted_win_rates,
        df_win_rates,
        df_kills,
        df_leads,
        df_tower_deaths,
        df_snapshots,
        df_tower_updates,
        df_outpost_updates,
        df_players
    ]
    second_layer_dfs = [
        df_performance_metrics, 
        df_death_events,
        df_inventory_reports,
        df_imp_per_minute,
        df_farm,
        df_buffs,
        df_purchases,
        df_courier_kills,
        df_runes,
        df_wards,
        df_ward_destructions
    ]
    if match_id == match_ids[0]:
        ## If it's the first result, create the tables
        dbf.create_table_from_df(df_match_details, 'match_details', conn_str=conn_str)
        dbf.create_table_from_df(df_pickbans, 'match_pick_bans', conn_str=conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_chatevents, 'match_chat_events', conn_str=conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_predicted_win_rates, 'match_predicted_win_rates', conn_str=conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_win_rates, 'match_win_rates', conn_str=conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_kills, 'match_kills', conn_str=conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_leads, 'match_leads', conn_str=conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_tower_deaths, 'match_tower_deaths', conn_str=conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_snapshots, 'match_snapshots', conn_str=conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_tower_updates, 'match_tower_updates', conn_str=conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_outpost_updates, 'match_outpost_updates', conn_str=conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_players, 'match_players', conn_str=conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_performance_metrics, 'match_performance_metrics', conn_str=conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_death_events, 'match_death_events', conn_str=conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_inventory_reports, 'match_inventory_reports', conn_str=conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_imp_per_minute, 'match_imp_per_minute', conn_str=conn_str, add_serial_id=True)
        df_farm.insert(0, 'farm_id', range(len(df_farm)))
        dbf.create_table_from_df(df_farm, 'match_farm', conn_str=conn_str)
        df_farm.drop('farm_id', axis=1, inplace=True)
        dbf.create_table_from_df(df_buffs, 'match_buffs', conn_str=conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_purchases, 'match_purchases', conn_str=conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_courier_kills, 'match_courier_kills', conn_str=conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_runes, 'match_runes', conn_str=conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_wards, 'match_wards', conn_str=conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_ward_destructions, 'match_ward_destructions', conn_str=conn_str, add_serial_id=True)
    dbf.insert_df_into_table(df_match_details, 'match_details', conn_str=conn_str)

    dbf.insert_df_into_table(df_pickbans, 'match_pick_bans', conn_str=conn_str)

    dbf.insert_df_into_table(df_chatevents, 'match_chat_events', conn_str=conn_str)

    dbf.insert_df_into_table(df_predicted_win_rates, 'match_predicted_win_rates', conn_str=conn_str)

    dbf.insert_df_into_table(df_win_rates, 'match_win_rates', conn_str=conn_str)

    dbf.insert_df_into_table(df_kills, 'match_kills', conn_str=conn_str)

    dbf.insert_df_into_table(df_leads, 'match_leads', conn_str=conn_str)

    dbf.insert_df_into_table(df_tower_deaths, 'match_tower_deaths', conn_str=conn_str)

    dbf.insert_df_into_table(df_snapshots, 'match_snapshots', conn_str=conn_str)

    dbf.insert_df_into_table(df_tower_updates, 'match_tower_updates', conn_str=conn_str)

    dbf.insert_df_into_table(df_outpost_updates, 'match_outpost_updates', conn_str=conn_str)

    dbf.insert_df_into_table(df_players, 'match_players', conn_str=conn_str)

    dbf.insert_df_into_table(df_performance_metrics, 'match_performance_metrics', conn_str=conn_str)

    dbf.insert_df_into_table(df_death_events, 'match_death_events', conn_str=conn_str)

    dbf.insert_df_into_table(df_inventory_reports, 'match_inventory_reports', conn_str=conn_str)

    dbf.insert_df_into_table(df_imp_per_minute, 'match_imp_per_minute', conn_str=conn_str)

    dbf.insert_df_into_table(df_farm, 'match_farm', conn_str=conn_str)

    dbf.insert_df_into_table(df_buffs, 'match_buffs', conn_str=conn_str)

    dbf.insert_df_into_table(df_purchases, 'match_purchases', conn_str=conn_str)

    dbf.insert_df_into_table(df_courier_kills, 'match_courier_kills', conn_str=conn_str)

    dbf.insert_df_into_table(df_runes, 'match_runes', conn_str=conn_str)

    dbf.insert_df_into_table(df_wards, 'match_wards', conn_str=conn_str)

    dbf.insert_df_into_table(df_ward_destructions, 'match_ward_destructions', conn_str=conn_str)
    unfinished_match_ids.remove(match_id)
    elapsed = time.time() - start_time
    if elapsed < 2.0:
        time.sleep(2.0-elapsed)


C:\Users\benib\AppData\Local\Temp\ipykernel_7196\3242279911.py:138: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_inventory_reports = pd.concat([df_inventory_reports, df_ir])
C:\Users\benib\AppData\Local\Temp\ipykernel_7196\3242279911.py:155: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_performance_metrics = pd.concat([df_performance_metrics, df_pm])
C:\Users\benib\AppData\Local\Temp\ipykernel_7196\3242279911.py:209: FutureWarning: The behavior of DataFrame concatenation with empty or all

Data inserted into table 'match_details' successfully.
Data inserted into table 'match_pick_bans' successfully.
Data inserted into table 'match_chat_events' successfully.
Data inserted into table 'match_predicted_win_rates' successfully.
Data inserted into table 'match_win_rates' successfully.
Data inserted into table 'match_kills' successfully.
Data inserted into table 'match_leads' successfully.
Data inserted into table 'match_tower_deaths' successfully.
Data inserted into table 'match_snapshots' successfully.
Data inserted into table 'match_tower_updates' successfully.
Data inserted into table 'match_outpost_updates' successfully.
Data inserted into table 'match_players' successfully.
Data inserted into table 'match_performance_metrics' successfully.
Data inserted into table 'match_death_events' successfully.
Data inserted into table 'match_inventory_reports' successfully.
Data inserted into table 'match_imp_per_minute' successfully.
Data inserted into table 'match_farm' successfull

C:\Users\benib\AppData\Local\Temp\ipykernel_7196\3242279911.py:138: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_inventory_reports = pd.concat([df_inventory_reports, df_ir])
C:\Users\benib\AppData\Local\Temp\ipykernel_7196\3242279911.py:155: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_performance_metrics = pd.concat([df_performance_metrics, df_pm])
C:\Users\benib\AppData\Local\Temp\ipykernel_7196\3242279911.py:209: FutureWarning: The behavior of DataFrame concatenation with empty or all

Data inserted into table 'match_details' successfully.
Data inserted into table 'match_pick_bans' successfully.
Data inserted into table 'match_chat_events' successfully.
Data inserted into table 'match_predicted_win_rates' successfully.
Data inserted into table 'match_win_rates' successfully.
Data inserted into table 'match_kills' successfully.
Data inserted into table 'match_leads' successfully.
Data inserted into table 'match_tower_deaths' successfully.
Data inserted into table 'match_snapshots' successfully.
Data inserted into table 'match_tower_updates' successfully.
Data inserted into table 'match_outpost_updates' successfully.
Data inserted into table 'match_players' successfully.
Data inserted into table 'match_performance_metrics' successfully.
Data inserted into table 'match_death_events' successfully.
Data inserted into table 'match_inventory_reports' successfully.
Data inserted into table 'match_imp_per_minute' successfully.
Data inserted into table 'match_farm' successfull

C:\Users\benib\AppData\Local\Temp\ipykernel_7196\3242279911.py:138: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_inventory_reports = pd.concat([df_inventory_reports, df_ir])
C:\Users\benib\AppData\Local\Temp\ipykernel_7196\3242279911.py:155: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_performance_metrics = pd.concat([df_performance_metrics, df_pm])
C:\Users\benib\AppData\Local\Temp\ipykernel_7196\3242279911.py:209: FutureWarning: The behavior of DataFrame concatenation with empty or all

...

Data inserted into table 'match_details' successfully.
Data inserted into table 'match_pick_bans' successfully.
Data inserted into table 'match_chat_events' successfully.
Data inserted into table 'match_predicted_win_rates' successfully.
Data inserted into table 'match_win_rates' successfully.
Data inserted into table 'match_kills' successfully.
Data inserted into table 'match_leads' successfully.
Data inserted into table 'match_tower_deaths' successfully.
Data inserted into table 'match_snapshots' successfully.
Data inserted into table 'match_tower_updates' successfully.
Data inserted into table 'match_outpost_updates' successfully.
Data inserted into table 'match_players' successfully.
Data inserted into table 'match_performance_metrics' successfully.
Data inserted into table 'match_death_events' successfully.
Data inserted into table 'match_inventory_reports' successfully.
Data inserted into table 'match_imp_per_minute' successfully.
Data inserted into table 'match_farm' successfull

C:\Users\benib\AppData\Local\Temp\ipykernel_7196\3242279911.py:138: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_inventory_reports = pd.concat([df_inventory_reports, df_ir])
C:\Users\benib\AppData\Local\Temp\ipykernel_7196\3242279911.py:155: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_performance_metrics = pd.concat([df_performance_metrics, df_pm])
C:\Users\benib\AppData\Local\Temp\ipykernel_7196\3242279911.py:209: FutureWarning: The behavior of DataFrame concatenation with empty or all

Data inserted into table 'match_details' successfully.
Data inserted into table 'match_pick_bans' successfully.
Data inserted into table 'match_chat_events' successfully.
Data inserted into table 'match_predicted_win_rates' successfully.
Data inserted into table 'match_win_rates' successfully.
Data inserted into table 'match_kills' successfully.
Data inserted into table 'match_leads' successfully.
Data inserted into table 'match_tower_deaths' successfully.
Data inserted into table 'match_snapshots' successfully.
Data inserted into table 'match_tower_updates' successfully.
Data inserted into table 'match_outpost_updates' successfully.
Data inserted into table 'match_players' successfully.
Data inserted into table 'match_performance_metrics' successfully.
Data inserted into table 'match_death_events' successfully.
Data inserted into table 'match_inventory_reports' successfully.
Data inserted into table 'match_imp_per_minute' successfully.
Data inserted into table 'match_farm' successfull

C:\Users\benib\AppData\Local\Temp\ipykernel_7196\3242279911.py:138: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_inventory_reports = pd.concat([df_inventory_reports, df_ir])
C:\Users\benib\AppData\Local\Temp\ipykernel_7196\3242279911.py:155: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_performance_metrics = pd.concat([df_performance_metrics, df_pm])
C:\Users\benib\AppData\Local\Temp\ipykernel_7196\3242279911.py:209: FutureWarning: The behavior of DataFrame concatenation with empty or all

Data inserted into table 'match_details' successfully.
Data inserted into table 'match_pick_bans' successfully.
Data inserted into table 'match_chat_events' successfully.
Data inserted into table 'match_predicted_win_rates' successfully.
Data inserted into table 'match_win_rates' successfully.
Data inserted into table 'match_kills' successfully.
Data inserted into table 'match_leads' successfully.
Data inserted into table 'match_tower_deaths' successfully.
Data inserted into table 'match_snapshots' successfully.
Data inserted into table 'match_tower_updates' successfully.
Data inserted into table 'match_outpost_updates' successfully.
Data inserted into table 'match_players' successfully.
Data inserted into table 'match_performance_metrics' successfully.
Data inserted into table 'match_death_events' successfully.
Data inserted into table 'match_inventory_reports' successfully.
Data inserted into table 'match_imp_per_minute' successfully.
Data inserted into table 'match_farm' successfull

In [29]:
result_json['predictedWinRates']

In [ ]:
nested_vars = []
for key, value in result_json.items():
    if type(value) in [dict, list]:
        print(key, type(value))
        nested_vars.append(key)

pickBans <class 'list'>
chatEvents <class 'list'>
predictedWinRates <class 'list'>
winRates <class 'list'>
radiantNetworthLeads <class 'list'>
radiantExperienceLeads <class 'list'>
radiantKills <class 'list'>
direKills <class 'list'>
towerDeaths <class 'list'>
towerStatus <class 'list'>
players <class 'list'>


In [ ]:
for key, value in result_json['players'][0].items():
    if type(value) in [dict, list]:
        print(key, type(value))
        nested_vars.append(key)

stats <class 'dict'>


In [25]:
for key, value in result_json['players'][0]['stats'].items():
    if type(value) in [dict, list]:
        print(key, type(value))
        nested_vars.append(key)

impPerMinute <class 'list'>
goldPerMinute <class 'list'>
networthPerMinute <class 'list'>
experiencePerMinute <class 'list'>
towerDamagePerMinute <class 'list'>
campStack <class 'list'>
deathEvents <class 'list'>
farmDistributionReport <class 'dict'>
matchPlayerBuffEvent <class 'list'>
inventoryReport <class 'list'>
itemPurchases <class 'list'>
courierKills <class 'list'>
runes <class 'list'>
wards <class 'list'>
wardDestruction <class 'list'>


In [51]:
mid_query = 'SELECT DISTINCT match_id FROM match_players WHERE match_players."steamAccountId" IS NULL'
matches = dbf.query_select_to_df(conn_str, mid_query, 'match_players', ['match_id'])
query = '''
    query($id: Long!) {
        match(id: $id) {
            players {
                heroId
                steamAccountId
                partyId
                steamAccount {
                    name
                    realName
                    profileUri
                    timeCreated
                    isAnonymous
                    proSteamAccount {
                        teamId
                        name
                    }
                }
            }
        }
    }
'''
with psycopg.connect(conn_str) as conn:
    with conn.cursor() as cur:
        for match_id in matches['match_id']:
            result = dbf.query_stratz(query, headers, api_url, variables={'id': match_id})
            res = result['data']['match']['players']
            df_players_addition = pd.DataFrame(res)
            df_steam_account = pd.json_normalize(df_players_addition['steamAccount'])
            og_cols = df_steam_account.columns
            new_cols = [str.replace(colname, '.', '_') for colname in df_steam_account.columns]
            col_mapper = {og_col: new_col for og_col, new_col in zip(og_cols, new_cols)}
            df_steam_account = df_steam_account.rename(col_mapper, axis=1)
            df_players_addition = df_players_addition.drop('steamAccount', axis=1)
            df_players_final = pd.concat([df_players_addition, df_steam_account], axis=1)
            try:
                df_players_final = df_players_final.drop('proSteamAccount', axis=1) #Sometimes this column gets left there empty
            except:
                pass
            for idx, row in df_players_final.iterrows():
                for col in df_players_final.columns:
                    if pd.isna(row[col]):
                        cur.execute(f'UPDATE match_players SET "{col}" = NULL WHERE match_players.match_id = %s AND match_players."heroId" = %s;', (match_id, row['heroId']))
                    else:
                        cur.execute(f'UPDATE match_players SET "{col}" = %s WHERE match_players.match_id = %s AND match_players."heroId" = %s;', (row[col], match_id, row['heroId']))
            conn.commit()
            

KeyError: 'data'

In [48]:
df_players_final

,heroId,steamAccountId,partyId,name,realName,profileUri,timeCreated,isAnonymous,proSteamAccount_teamId,proSteamAccount_name
0,28,116293223,None,Tempestira,,https://steamcommunity.com/id/76561198076558951/,NaN,True,8740972.0,Black Soul
1,128,133958007,None,☼☼,,https://steamcommunity.com/id/76561198094223735/,1.370974e+09,False,8272699.0,Hanzama <
2,75,99796146,None,"Natanael:,",,https://steamcommunity.com/id/starGazerCactusA...,NaN,True,8856662.0,Accel
3,102,307524380,None,X.x-Lw,,https://steamcommunity.com/id/307524380/,NaN,True,NaN,NaN
4,73,173842160,None,)(,,https://steamcommunity.com/id/76561198134107888/,NaN,True,NaN,NaN
5,87,248180032,None,Four,,https://steamcommunity.com/id/76561198208445760/,1.422453e+09,False,8375259.0,Michael-
6,120,1131665504,None,20K,,https://steamcommunity.com/id/76561199091931232/,NaN,True,8971308.0,Ñengoflow KELOKE
7,64,288144026,None,CHEFCITO,,https://steamcommunity.com/id/76561198248409754/,NaN,True,NaN,NaN
8,46,182322692,None,SWAP COMEND,,https://steamcommunity.com/id/76561198142588420/,1.403992e+09,False,NaN,NaN
9,33,125845797,None,Timothy,,https://steamcommunity.com/id/23456ty789203213...,NaN,True,0.0,RedMoon


In [47]:
row[col]

nan

In [46]:
col

'proSteamAccount_teamId'

In [34]:
with psycopg.connect(conn_str) as conn:
    cols = []
    for col_name, dtype in zip(df_players_final.columns, df_players_final.dtypes):
        if col_name != 'heroId':
            pg_type = dbf.get_pg_type(dtype)
            # Wrap column names in quotes to handle spaces or reserved words
            cols.append(f'ADD COLUMN"{col_name}" {pg_type}')
    
    schema = ", ".join(cols)
    create_table_query = f'ALTER TABLE "match_players" {schema};'
    with conn.cursor() as cur:
        cur.execute(create_table_query)